# GrootCV objectives, boosting limits and custom splits

Start with a small binary classification fixture. Change one mechanism at
a time and inspect what the model actually used. All fits below use the
training frame only.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

from arfs.feature_selection import GrootCV

rng = np.random.default_rng(42)
X = pd.DataFrame(rng.normal(size=(300, 3)), columns=["signal", "noise_a", "noise_b"])
y = (X["signal"] > 0).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
options = {"n_folds": 3, "n_iter": 1, "n_jobs": 1, "random_state": 42}

## Objective precedence

A supplied `objective` argument overrides `lgbm_params["objective"]`.
With `objective=None`, GrootCV reads the dictionary. Here both fits resolve
to binary classification and use the same default stratified folds.

In [2]:
direct = GrootCV(
    objective="binary",
    lgbm_params={"objective": "regression", "num_iterations": 12},
    **options,
).fit(X_train, y_train)
from_params = GrootCV(
    lgbm_params={"objective": "binary", "num_iterations": 12},
    **options,
).fit(X_train, y_train)
pd.testing.assert_frame_equal(direct.cv_df, from_params.cv_df)
direct.cv_df[["feature", "Med"]]

,feature,Med
0,ShadowVar1,0.000000e+00
1,ShadowVar2,0.000000e+00
2,ShadowVar3,0.000000e+00
3,noise_a,7.523406e-18
4,noise_b,0.000000e+00
5,signal,1.000000e+00


## Boosting limits

Set `num_iterations` in `lgbm_params` to bound training in every fold.
LightGBM can stop early with supported boosting types. The fit below
prints the best iteration per fold. A DART fit needs an explicit limit
because it cannot use early stopping.

In [3]:
bounded = GrootCV(
    objective="binary",
    silent=False,
    lgbm_params={"num_iterations": 6},
    **options,
).fit(X_train, y_train)
bounded.selected_features_

GrootCV: classification with unbalance classes


Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[6]	training's binary_logloss: 0.313165	valid_1's binary_logloss: 0.313291
Groot iteration:  0  with 3 folds
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[6]	training's binary_logloss: 0.313165	valid_1's binary_logloss: 0.313291
Training until validation scores don't improve for 20 rounds
Did not meet early stopping. Best iteration is:
[6]	training's binary_logloss: 0.313332	valid_1's binary_logloss: 0.313206


array(['signal'], dtype=object)

## Custom splits

`folds` takes precedence over `n_folds` and `n_iter`. This reusable list
contains two train/validation pairs with positions relative to the training
frame. Use a matching splitter for grouped or chronological data. If you
supply a generator, create a new one before refitting.

In [4]:
splitter = StratifiedKFold(n_splits=2, shuffle=True, random_state=7)
splits = list(splitter.split(X_train, y_train))
custom = GrootCV(
    objective="binary",
    lgbm_params={"num_iterations": 12},
    folds=splits,
    **options,
).fit(X_train, y_train)
fold_scores = custom.cv_df.filter(like="fscore")
assert fold_scores.shape[1] == len(splits)
display(pd.concat([custom.cv_df["feature"], fold_scores], axis=1))
print("Held-out shape:", custom.transform(X_test).shape)

,feature,fscore0,fscore1
0,ShadowVar1,0.0,0.000000e+00
1,ShadowVar2,0.0,0.000000e+00
2,ShadowVar3,0.0,0.000000e+00
3,noise_a,0.0,3.067492e-18
4,noise_b,0.0,0.000000e+00
5,signal,1.0,1.000000e+00


Held-out shape: (75, 2)
